# Un nuevo objetivo: Predicción 


En el pasado hemos trabajado con la regresión lineal tratando de modelar la relación entre un conjunto de variables independientes $\mathbb{X}$ y una variable de respuesta $\mathbf{Y}$. Primero fue de nuestro interés entender cómo las variables independientes afectan a la variable de respuesta y verificar preguntas específicas del contexto de los datos, es decir, nos enfocamos en realizar **inferencia** sobre el modelo de regresión, mediante estimaciones puntuales, intervalos de confianza y pruebas de hipótesis sobre los coeficientes de regresión.

Sin embargo, en muchos casos nuestro interés principal es **predecir** el valor de la variable de respuesta $\mathbf{Y}$ para nuevos valores de las variables independientes $\mathbb{X}$. Para ello, buscamos a partir de las $n$ observaciones disponibles, construir un modelo que nos permita predecir el valor de **$y$** para nuevos valores de **$x_{1}, ... , x_{p}$**.

En resumen, en el enfoque de inferencia el objetivo era entender qué causa tener un valor $y$ especifico, pero para el enfoque de predicción, el objetivo es construir una regla que permita predecir $y$ sin necesidad de entender completamente las causas que lo generan.

Por ejemplo, en un contexto médico, para predecir si un paciente tiene una infección viral, se puede construir una regla basada en características clínicas sin necesidad de entender completamente el mecanismo de contagio del virus.

**Para escoger un buen modelo predictivo, es fundamental evaluar su desempeño en nuevas observaciones (que no hayan sido utilizadas durante el proceso de ajuste del modelo) y compararlo con modelos alternativos.**


## **Etapas**

### 1. **Entrenamiento del modelo (Construir la regla/modelo)** 

**Se utiliza el conjunto de $n$ observaciones disponibles para ajustar el modelo predictivo.** 

Aquí se seleccionan las variables independientes y se estiman los coeficientes del modelo.

Luego, se puede obtener su poder predictivo **aparente**, es decir, **el desempeño del modelo al predecir las mismas observaciones que se usaron para ajustarlo**; este valor suele ser optimista y no refleja el desempeño real del modelo en nuevas observaciones, ya que no se está evaluando su capacidad de predecir para nuevos valores de **$x_{1}, ... , x_{p}$**. 

Realizar esta evaluación permite descartar modelos que no modelan correctamente el fenómeno de interés y seleccionar aquellos que parecen tener un buen desempeño predictivo, pues si el poder predictivo aparente es bajo, el modelo seguramente no tendrá un buen desempeño en nuevas observaciones.

En la regresión lineal, el **poder predictivo aparente se puede medir mediante indices como el MSE (Mean Squared Error) o el MAE (Mean Absolute Error)calculados sobre las mismas observaciones utilizadas para ajustar el modelo**. 

### 2. **Evaluación del poder predictivo en nuevas observaciones** 



Una vez que se tiene **la forma del modelo que no depende de las observaciones específicas** (por ejemplo, las variables que entran al modelo o el uso de metodos de selección de variables como Ridge, Lasso o stepwise) , es necesario evaluar su poder predictivo en nuevas observaciones **que no hayan sido utilizadas durante el ajuste del modelo.**

Para lograr una comparativa entre los valores predichos por un modelo y los valores reales de la variable de respuesta, se opta por dividir el conjunto de datos disponible en dos partes: un conjunto de entrenamiento (**train**) y un conjunto de prueba (**test**).

 **El conjunto train se utiliza para ajustar el modelo (ESTIMAR COEFICIENTES), mientras que el conjunto test se emplea para comparar las predicciones del modelo con los valores REALES Y CONOCIDOS de la variable de respuesta.**

Esta propuesta tiene la principal desventaja de que al dividir los datos, existe la posibilidad de que el modelo esté sesgado hacia las observaciones del conjunto de entrenamiento, lo que puede afectar su capacidad de generalización a nuevas observaciones.

Por lo que deseable que:

1. El conjunto test represente todas las características que en la práctica pueden presentar las nuevas observaciones.

2. El conjunto train no sea tan pequeño, o en su defecto, que la regla que se está evaluando no sea muy diferente de la regla que se obtiene sólo con el conjunto train. (i.e los coeficientes del modelo ajustado con n observaciones no sean muy diferentes a los coeficientes del modelo ajustado con el conjunto train).

La forma en que se "mitigará" este problema es mediante alguno de estos métodos:

- **Repeated Holdout**:

Consiste en fijar las proporciones del conjunto de entrenamiento y de prueba (usualmente 80% para train y 20% para test), y realizar $B$ ocasiones una estimación del poder predictivo.

En cada ocasión se realiza un muestreo aleatorio para escoger al conjunto de entrenamiento (y el complemento es el de prueba).

Finalmente, se realiza el promedio de los $B$ valores calculados.


- **K cross-validation**: 
 
 Consiste en dividir aleatoriamente el conjunto de datos en $K$ subconjuntos (folds) de tamaño similar. Se recomienda que $K$ tome valores entre 5 y 10.
 
 Luego, se realizan $K$ iteraciones donde en cada iteración se utiliza un fold como el conjunto test y los otros K-1 folds como el conjunto train. 
  
 Se ajusta el modelo con el conjunto train y se evalúa su desempeño en el conjunto test (con MSE o MAE). 
 
 Finalmente, se promedian los resultados obtenidos en las $K$ iteraciones para obtener una estimación del poder predictivo del modelo.


- **Repeated K cross-validation**:

Consiste en repetir el proceso de K cross-validation $B$ veces (cada vez con una división diferente de los datos en folds pues se realiza de forma aleatoria).

Y promediar los resultados obtenidos en las $B$ repeticiones para obtener la estimación del poder predictivo del modelo.

Obs. **En total, se usan $B$ muestras, y cada una de ellas contiene $K$ submuestras** 


















# Ejemplo

Usaremos la base de datos `Hitters` del paquete `ISLR`, que contiene información sobre las estadísticas de bateo y los salarios de jugadores de béisbol de las ligas mayores. **Nuestro objetivo es predecir el salario de un jugador en función de sus estadísticas de bateo.** La métrica a optimizar es $MSE$.

Por simplicidad, eliminaremos las observaciones con valores faltantes. 

La variable respuesta es `Salary` y las variables predictoras son las demás variables numéricas en el conjunto de datos.


In [3]:
library(ISLR)

# diccionario de datos
# help("Hitters")

Datos=Hitters
Datos = na.omit(Datos)
str(Datos)

Warning message:
"package 'ISLR' was built under R version 4.5.2"


'data.frame':	263 obs. of  20 variables:
 $ AtBat    : int  315 479 496 321 594 185 298 323 401 574 ...
 $ Hits     : int  81 130 141 87 169 37 73 81 92 159 ...
 $ HmRun    : int  7 18 20 10 4 1 0 6 17 21 ...
 $ Runs     : int  24 66 65 39 74 23 24 26 49 107 ...
 $ RBI      : int  38 72 78 42 51 8 24 32 66 75 ...
 $ Walks    : int  39 76 37 30 35 21 7 8 65 59 ...
 $ Years    : int  14 3 11 2 11 2 3 2 13 10 ...
 $ CAtBat   : int  3449 1624 5628 396 4408 214 509 341 5206 4631 ...
 $ CHits    : int  835 457 1575 101 1133 42 108 86 1332 1300 ...
 $ CHmRun   : int  69 63 225 12 19 1 0 6 253 90 ...
 $ CRuns    : int  321 224 828 48 501 30 41 32 784 702 ...
 $ CRBI     : int  414 266 838 46 336 9 37 34 890 504 ...
 $ CWalks   : int  375 263 354 33 194 24 12 8 866 488 ...
 $ League   : Factor w/ 2 levels "A","N": 2 1 2 2 1 2 1 2 1 1 ...
 $ Division : Factor w/ 2 levels "E","W": 2 2 1 1 2 1 2 2 1 1 ...
 $ PutOuts  : int  632 880 200 805 282 76 121 143 0 238 ...
 $ Assists  : int  43 82 11 40 42

Exploraremos los siguientes modelos de regresión lineal:

1. Modelo de efectos principales

2. Modelo solo con las variables continuas al cuadrado.

Para obtener los valores predichos por un modelo, usaremos `predict()`

```{r}
    # predecir los valores con los que se entreno el modelo
    predictions_train <- predict(model_object)

    # predecir nuevas observaciones
    predictions_new <- predict(model_object, newdata = new_data_frame)

```

## Entrenamiento del modelo (Construir la regla/modelo)

### Modelo 1: Efectos principales

In [ ]:
# relga 
mod1 <- lm(Salary ~ ., data = Datos) 

# poder predictivo aparente MSE

predicciones1 <- predict(mod1, newdata = Datos)

aparente1 <- mean((Datos$Salary - predicciones1)^2)

aparente1


[1] 92017.87

### Modelo 2: Solo variables continuas al cuadrado

Primero hay que identificar las variables continuas en el conjunto de datos.

Luego, se crea una formula de R que incluya solo las variables continuas al cuadrado.

Es importante destacar que usar una fórmula solo con las variables continuas al cuadrado no depende de las observaciones que se tengan.

In [ ]:
#todas las continuas
(continuas = names(which(sapply(Datos, is.numeric))) )

# quitamos la variable respuesta
xnames=continuas[!continuas %in%  c("Salary")]

# creamos la formula
forexp=as.formula(  paste('Salary ~.',"+", paste(paste('I(',xnames,'^2)',collapse = ' + ')  ) )) 
forexp

# regla
mod2=lm(forexp, Datos) 


# poder predictivo aparente MSE
predicciones2 <- predict( mod2, newdata = Datos)
aparente2 <- mean((Datos$Salary - predicciones2)^2)
aparente2

[1] "AtBat"   "Hits"    "HmRun"   "Runs"    "RBI"     "Walks"   "Years"  
 [8] "CAtBat"  "CHits"   "CHmRun"  "CRuns"   "CRBI"    "CWalks"  "PutOuts"
[15] "Assists" "Errors"  "Salary"

Salary ~ . + I(AtBat^2) + I(Hits^2) + I(HmRun^2) + I(Runs^2) + 
    I(RBI^2) + I(Walks^2) + I(Years^2) + I(CAtBat^2) + I(CHits^2) + 
    I(CHmRun^2) + I(CRuns^2) + I(CRBI^2) + I(CWalks^2) + I(PutOuts^2) + 
    I(Assists^2) + I(Errors^2)

[1] 52488.89

### Comparativas de las medidas aparentes de poder predictivo

Parece que el modelo 2 tiene un mejor poder predictivo aparente que el modelo 1, y aunque esto no garantiza que el modelo 2 tenga un mejor desempeño en nuevas observaciones, es un indicio de que podría ser así.

In [6]:
aparente1 < aparente2

[1] FALSE

## Evaluación del poder predictivo en nuevas observaciones

Por simplicidad usaremos **Repeated holdout** con $B=50$ y la proporción de datos de entrenamiento es 80%.

Esto quiere decir que:

Para seleccionar el conjunto de entrenamiento, se tomarán de forma aleatoria observaciones hasta tener la proporción del 80%. 

Se ajusta el modelo y calcula el poder preditivo.

Se realiza el procedimiento anterior 10 veces, y se promedian las medidas de poder predictivo.



Definamos las "muestras" de entrenamiento.

La función `createDataPartition()` de la libreria `caret` devuelve un arreglo de indices.

In [8]:
library(caret)

set.seed(1)

B<-50

particion <- createDataPartition(Datos$Salary, p = .8, groups =4, list = FALSE, times = B)


# como n=263, y se toma el 80% como entrenamiento
# 0.8 * 263 = 210.4 observaciones de entrenamiento

dim(particion)
head(particion)

[1] 212  50

Resample01,Resample02,Resample03,Resample04,Resample05,Resample06,Resample07,Resample08,Resample09,Resample10,⋯,Resample41,Resample42,Resample43,Resample44,Resample45,Resample46,Resample47,Resample48,Resample49,Resample50
1,1,1,2,1,1,2,1,2,1,⋯,1,1,1,2,1,1,1,2,1,2
2,2,2,3,2,2,3,2,3,2,⋯,2,2,3,3,3,2,2,3,2,3
3,3,3,4,4,3,6,3,4,4,⋯,4,3,4,5,4,3,4,4,3,5
4,4,4,5,5,6,7,5,5,6,⋯,5,5,5,6,5,5,5,5,5,6
5,5,5,7,6,7,8,6,6,7,⋯,6,6,6,7,6,6,8,6,6,7
6,6,6,8,7,8,10,8,7,8,⋯,7,7,7,8,8,8,9,7,7,8


Definimos las reglas. Conviene que sean funciones pues se aplicarán para cada muestra.

In [9]:
# regla 1

#modelo 1 con Repeated Holdout

mod1RH=function(x, muestras_totales, datos){
  # x es la iteracion (B veces)
  # muestras_totales es un arreglo donde la i-esima columna es la i-esima muestra (B muestras)
  # datos son los datos originales

  train= muestras_totales[,x]
  test = (-train)
  modtr=lm(Salary ~ ., datos[train,])
  predte=predict(modtr, datos[test,])
  MSE=mean((datos$Salary[test]-predte)^2)
  return(MSE)
  
}



MSE.B.mod1= sapply(1:B,mod1RH, muestras_totales=particion, datos=Datos)
(MSE.RHM.mod1=mean(MSE.B.mod1))




[1] 118964.9

In [10]:
# regla 2

# modelo 2 con repeated holdout

mod2RH=function(x, muestras_totales, datos, formula){
  # x es la iteracion (B veces)
  # muestras_totales es un arreglo donde la i-esima columna es la i-esima muestra (B muestras)
  # datos son los datos originales
  # formula es la formula especifica que se ajusta en la regresion, NO DEPENDE DE LAS OBSERVACIONES

  train= muestras_totales[,x]
  test = (-train)
  modtr=lm(formula, datos[train,])
  predte=predict(modtr, datos[test,])
  MSE=mean((datos$Salary[test]-predte)^2)
  return(MSE)
}


MSE.B.mod2= sapply(1:B,mod2RH, muestras_totales=particion, datos=Datos, formula=forexp)
(MSE.RHM.mod2=mean(MSE.B.mod2))


[1] 124168.7

Para valores pequeños de $B$, parece que tiene un mejor poder predictivo el modelo de efectos principales.

¿Qué pasaría si $B$ crece?

In [11]:
B<- 10000

particion <- createDataPartition(Datos$Salary, p = .8, groups =4, list = FALSE, times = B)


MSE.B.mod1= sapply(1:B,mod1RH, muestras_totales=particion, datos=Datos)
(MSE.RHM.mod1=mean(MSE.B.mod1))




[1] 121247.6

In [12]:
MSE.B.mod2= sapply(1:B,mod2RH, muestras_totales=particion, datos=Datos, formula=forexp)
(MSE.RHM.mod2=mean(MSE.B.mod2))

[1] 130631.2

# Métodos de selección de variables